# BAB VI. Transformasi dan Rekayasa Fitur (Data Transformation and Feature Engineering)

Transformasi dilakukan pada salinan kerja, bukan pada data mentah. Tentukan kolom dan format berdasarkan kamus data yang telah diverifikasi. Tidak ada konversi otomatis atas seluruh kolom.

In [ ]:
from pathlib import Path
import pandas as pd
from eda_knkt.config import PROCESSED_DATA_DIR
from eda_knkt.ingestion import load_csv
from eda_knkt.cleaning import replace_placeholders
from eda_knkt.transformation import convert_numeric_columns, add_distance_to_stop_feature

DATA_FILE = PROCESSED_DATA_DIR / 'data_logger_knkt_tabel_6_10.csv'
df_raw = load_csv(DATA_FILE)
df_missing = replace_placeholders(df_raw, ['---'])
NUMERIC_OPERATIONAL_COLUMNS = [
    'km', 'm', 'Speed (km/h)', 'Acceleration (m/sec²)',
    'Load (amps)', 'EAB BP (psi)', 'EAB BC (psi)'
]
df_transformed = convert_numeric_columns(
    df_missing, NUMERIC_OPERATIONAL_COLUMNS, errors='coerce'
)
df_transformed = add_distance_to_stop_feature(df_transformed)
df_transformed['Timestamp'] = pd.to_datetime(
    df_transformed['Date'].astype(str) + ' ' + df_transformed['Time'].astype(str),
    format='%d/%m/%Y %H.%M.%S', errors='raise'
)
df_transformed = df_transformed.sort_values('Timestamp').reset_index(drop=True)
df_transformed['Elapsed (s)'] = (
    df_transformed['Timestamp'] - df_transformed['Timestamp'].iloc[0]
).dt.total_seconds()
print(f'Loaded {len(df_raw)} records × {len(df_raw.columns)} columns from {DATA_FILE.name}')

In [ ]:
from eda_knkt.transformation import (
    convert_numeric_columns,
    convert_datetime_columns,
    sort_dataframe,
    select_columns,
)

if "df_missing" not in globals():
    print("Jalankan notebook 03 dan 04 terlebih dahulu.")
else:
    df_transformed = df_missing.copy()
    print("Available columns:", df_transformed.columns.tolist())

    # Isi hanya setelah tipe dan format kolom diketahui dari sumber.
    VERIFIED_NUMERIC_COLUMNS = []
    VERIFIED_DATETIME_COLUMNS = []
    VERIFIED_DATETIME_FORMAT = None

    if VERIFIED_NUMERIC_COLUMNS:
        df_transformed = convert_numeric_columns(
            df_transformed, VERIFIED_NUMERIC_COLUMNS, errors="raise"
        )
    if VERIFIED_DATETIME_COLUMNS:
        df_transformed = convert_datetime_columns(
            df_transformed,
            VERIFIED_DATETIME_COLUMNS,
            format=VERIFIED_DATETIME_FORMAT,
            errors="raise",
        )
    display(df_transformed.dtypes.astype(str).to_frame("dtype"))

## Pilihan transformasi
Skewness transformation, scaling/normalization, categorical encoding, dan feature engineering hanya diterapkan jika relevan dengan tujuan analisis serta didukung makna variabel.

Pada EDA rekaman Data Logger:
- Jangan menerapkan log, akar kuadrat, Box-Cox, atau Yeo-Johnson tanpa memeriksa domain nilai dan tujuan.
- Scaling bukan keharusan untuk statistik deskriptif.
- Encoding kategori tidak diperlukan untuk visualisasi frekuensi dasar.
- Ekstraksi jam/hari hanya dilakukan setelah zona waktu, format, dan makna timestamp diverifikasi.
- Jangan menciptakan fitur rasio atau agregasi tanpa definisi teknis yang sah.

Catat kolom sumber, aturan, parameter, dan hasil sebelum/sesudah setiap transformasi.